# 06 — Predicción Final: Gran Premio de España 2026 (Madrid)

Este notebook consolida todas las decisiones del pipeline y genera la predicción final.

**Fecha de carrera:** 2026-09-13  
**Circuito:** Madring (nuevo — sin historia en F1)  
**Pole position:** Lando Norris (McLaren) — 1:31.824

### Decisiones metodológicas
- **Modelo:** Gradient Boosting (mejor Log Loss en walk-forward: 0.1072)
- **Ventana histórica:** determinada por window comparison
- **Anti-leakage:** qualifying_position viene del endpoint de qualifying (no de race results)
- **Normalización:** probabilidades softmax → suma = 100%

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from src.config import PROCESSED_DIR, FIGURES_DIR, TARGET
from src.features.build_features_combined import MODEL_FEATURE_COLUMNS
from src.modeling.model_registry import get_model_builders
from src.evaluation.metrics import predict_race_probabilities

In [ ]:
# Load results
features = pd.read_csv(PROCESSED_DIR / 'features_combined.csv')
target = pd.read_csv(PROCESSED_DIR / 'madring_2026_prediction_input.csv')
wf = pd.read_csv(PROCESSED_DIR / 'walk_forward_results.csv')

print(f'Training data: {len(features)} rows')
print(f'Target GP: {len(target)} drivers')
print()
print('Walk-forward summary:')
print(wf.groupby('model')[['log_loss','winner_accuracy']].mean().round(4).sort_values('log_loss'))

In [ ]:
# Try to load window comparison results
try:
    wc = pd.read_csv(PROCESSED_DIR / 'window_comparison_results.csv')
    best_window_row = wc.groupby(['window','model'])['log_loss'].mean().sort_values().index[0]
    best_window_label, best_model_name = best_window_row
    # Map window label to year
    window_map = {'3yr': 2023, '5yr': 2021, '7yr': 2019, 'all': 2018}
    best_window_start = window_map.get(best_window_label, 2018)
    print(f'Best window: {best_window_label} (start {best_window_start})')
    print(f'Best model:  {best_model_name}')
except FileNotFoundError:
    # Fall back to walk-forward winner
    best_model_name = wf.groupby('model')['log_loss'].mean().sort_values().index[0]
    best_window_start = 2018
    print(f'(Window comparison not found, using walk-forward winner)')
    print(f'Best model: {best_model_name}, window start: {best_window_start}')

In [ ]:
# Train final model on full history up to round 13 of 2026
train = features[
    (features['season'] >= best_window_start) &
    ~((features['season'] == TARGET['year']) & (features['round'] >= TARGET['round']))
].dropna(subset=['won'])

print(f'Final training set: {len(train)} rows, {train.groupby(["season","round"]).ngroups} races')
print(f'Seasons: {train["season"].min()} - {train["season"].max()}')

models = get_model_builders(MODEL_FEATURE_COLUMNS)
model = models[best_model_name]
model.fit(train[MODEL_FEATURE_COLUMNS], train['won'].astype(int))
print('Model trained successfully')

In [ ]:
# Generate predictions
predictions = predict_race_probabilities(model, target, MODEL_FEATURE_COLUMNS)
predictions['win_probability_pct'] = (predictions['win_probability'] * 100).round(1)
predictions['rank'] = range(1, len(predictions) + 1)

print(f'\n{"="*60}')
print(f'GRAN PREMIO DE ESPANA 2026 — MADRID')
print(f'Pre-race win probability prediction')
print(f'{"="*60}')
display_cols = ['rank', 'driver_id', 'constructor_id', 'qualifying_position', 'win_probability_pct']
print(predictions[display_cols].to_string(index=False))
print(f'\nTotal probability: {predictions["win_probability_pct"].sum():.1f}%')

In [ ]:
# Visualization
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

top_n = predictions.head(10)
colors_map = {
    'mclaren': '#FF8000', 'mercedes': '#00D2BE', 'red_bull': '#1E41FF',
    'ferrari': '#DC0000', 'alpine': '#0090FF', 'aston_martin': '#006F62',
    'rb': '#6692FF', 'haas': '#B6BABD', 'williams': '#005AFF',
    'audi': '#FF0000', 'cadillac': '#FFFFFF'
}
bar_colors = [colors_map.get(c, '#999999') for c in top_n['constructor_id']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    range(len(top_n)),
    top_n['win_probability_pct'].values,
    color=bar_colors,
    edgecolor='black', linewidth=0.5
)
ax.set_yticks(range(len(top_n)))
ax.set_yticklabels([
    f"{row['driver_id']} (P{int(row['qualifying_position'])})"
    for _, row in top_n.iterrows()
], fontsize=11)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=10)
ax.set_xlabel('Win probability (%)', fontsize=12)
ax.set_title(
    f'Pre-race win probability\nSpanish Grand Prix 2026 (Madring)',
    fontsize=13, fontweight='bold'
)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
out = FIGURES_DIR / 'final_prediction_madrid_2026.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
print(f'Chart saved -> {out}')
plt.show()

In [ ]:
# Save final predictions
out_csv = PROCESSED_DIR / f'final_prediction_madring_2026.csv'
save_cols = ['rank','driver_id','driver_code','constructor_id','qualifying_position',
             'gap_to_pole_sec','raw_win_probability','win_probability','win_probability_pct']
avail_cols = [c for c in save_cols if c in predictions.columns]
predictions[avail_cols].to_csv(out_csv, index=False)
print(f'Predictions saved -> {out_csv}')

print('\n=== PREDICTION SUMMARY ===')
print(f'Model: {best_model_name}')
print(f'Training window: {best_window_start} onwards')
print(f'Predicted winner: {predictions.iloc[0]["driver_id"]} ({predictions.iloc[0]["win_probability_pct"]:.1f}%)')
print(f'Qualifying position of predicted winner: P{int(predictions.iloc[0]["qualifying_position"])}')